I'm interested to see where people move to DC from.

I'd like to group the DMV -- DC, Maryland and Virginia -- together for this.

Relatedly, I'm currious whether in election years, the influx comes from redder/bluer states depending on election results.

In [ ]:
import pandas as pd
import numpy as np


Read in the file

In [ ]:
df = pd.read_excel('ORIGINAL DATA state_to_state_migrations_table_2009.xls')

In [ ]:
df.shape

In [ ]:
df.columns

Need to drop the notes in the excel doc that come before and after the actual chart.
I am also dropping the middle repeated headers columns 45-47 in the doc

In [ ]:

df = df.iloc[np.r_[4:43, 44:77]]
                   
df.head()



It looks like pandas has already automatically unmerged the header columns that in the .xls cover more than one column, so now I'm just going to forward fill the headers to avoid null values.

In [ ]:

#checking the columns, can see that this still needs work

df.columns


In [ ]:
df.head(20)

Drop rows that appear to be for readability of the .xls by dropping all rows that are NaN across the board.

In [ ]:
df = df.dropna(how="all")

df.head(20)

Rename columns to be clearer, including manually renaming certain columns and stripping 'Unnamed: ' from the rest

In [ ]:
print(df.columns.tolist())

In [ ]:
#for the first column

df.rename(columns={'Dataset: 2007-2009 American Community Survey 3-Year Estimates': '0'}, inplace=True)

#renaming all of these weird long composite names at the sart of every group of 11

df.rename(columns= {'Table with row headers in column A, L, W, AH, AS, BD, BO, BZ, CK, CV, and DG, and column headers in rows 6 through 8 and 46 through 48.': '11'}, inplace=True)
for i in range(1, 10):
    number = (i * 11) + 11
    df.rename(columns={f'Table with row headers in column A, L, W, AH, AS, BD, BO, BZ, CK, CV, and DG, and column headers in rows 6 through 8 and 46 through 48..{i}':f'{number}'}, inplace=True)

#for the rest of the columns, just trimming:

for col in df.columns[1:130]:  # 2–129 (0-based index, so skip first col)
    if str(col).startswith("Unnamed:"):
        df.rename(columns={col: str(col).replace("Unnamed: ", "")}, inplace=True)

print(df.columns.tolist())

In [ ]:
columns_to_drop =  ['11', '22', '33', '44', '55', '66', '77', '88', '99', '110']

for column in columns_to_drop:
    df.drop(columns = [column], inplace = True)

Drop the repeated "Current Residence In" columns that are there for readability (identical to the state titles in column A, they repeat in A, L, W, AH, AS, BD, BO, BZ, CK, CV, DG, DR)


Before I forward fill into the null values, I want to make sure to preserve any values that were listed as N/A in the original dataset.

There is one set for each of the states. Specifically, for those people that had a different state of residence last year than this year, the cells in the excel sheet where the row state (currently reside in) and the column state (lived last year) are the same are filled in as N/A.

Here, it makes sense to replace them with zero, because no people who had a different state of residence last year than this year lived in a given state both last year and this year.

In [ ]:
df.iloc[3:] = df.iloc[3:].fillna('0')

In [ ]:
df.head()

Rename columns by forwards filling some of what, in the original excel sheet, was merged column titles spanning more than one column

In [ ]:
#For the rest of the columns:
#first, fill in blanks

df_filledin = df.ffill(axis=1)

df_filledin.head()


In [ ]:
#then pull the info I need to name each column

for i in range(1, 105):
    new_column_name = f'{df_filledin.iloc[0, i]}: {df_filledin.iloc[1, i]} ({df_filledin.iloc[2, i]})'
    current_column_name = df_filledin.columns[i]
    df_filledin = df_filledin.rename(columns = {current_column_name: new_column_name})


In [ ]:
df_filledin.head()

Now that everything is properly renamed (ie the various titles and subtitles are brought into the title of the actual column), we can delete the subtitle rows

In [ ]:
df = df_filledin.iloc[3:]

df.tail(20)


Transposing for further cleaning

Using "set_index" to tell it to use the column titled 0 as the new column titles.

In [ ]:
df.columns

df_trans = df.set_index(df['0']).transpose()

df_trans

In [ ]:
df = df_trans

Now that its transposed, I just need to drop the first row that is now already in the names of the columns.

In [ ]:

df = df.iloc[1:]

In [ ]:
df

Wooohooo now this finally looks good.

Now we can analyze some stuff.

First up, find DC info:

In [ ]:
df = df[['District of Columbia ']]
# indices = df.index.tolist
# indices

Import population maps in order to adjust each state per capita

In [ ]:
df_pops = pd.read_csv('State populations 2020 from census.csv')

df_pops = df_pops.transpose()
df_pops

Strip out all columns except those representing the number estimate for inmigration from each state.

I'm doing this by saving the NaNs out for now, though, as I want to bring them back.

In [ ]:
df_estimates = df[df.index.astype(str).str.contains('Estimate', case=False, na=False)]

df_moes = df[df.index.astype(str).str.contains('MOE', case=False, na=False)]

df_estimates

Renaming all the indexes to just be the state names in both estimates and moes data frames so that I can then merge them.


In [ ]:
df_estimates.index = df_estimates.index.to_series().astype(str).str.extract(r':\s*(.*?)\s*\(')[0]

df_moes.index = df_moes.index.to_series().astype(str).str.extract(r':\s*(.*?)\s*\(')[0]



For 2021 and 2017, we needed to cut off the top columns that arent for specific states, but that's not relevent here because of different structure for 2009 and 2005

In [ ]:

# df_estimates = df_estimates[4:]
# df_moes = df_moes[4:]

In [ ]:
df_moes.head()


In [ ]:
df_estimates.head()

Add the moes information (currently its own dataframe) as a column into the estimates dataframe

In [ ]:
df_estimates['MOE +/-'] = df_moes['District of Columbia ']

df = df_estimates

Now I'm adding a state polulation column to my newly formatted table

In [ ]:
df['state_population'] = df_pops[0]

Rename the columns to make more sense

In [ ]:
df.rename(columns={'District of Columbia ':'pop_migrated'}, inplace=True)


And strip commas in state_population column

In [ ]:
df['state_population'] = df['state_population'].str.replace(',', '')

Checking progress

In [ ]:
df

Add a row with the proportion of state residents counted in the census that moved to DC (ie divide each state's immigration to DC number by the total pop of the state sending those people to DC)

In [ ]:
df['migrated_adjusted'] = (
    pd.to_numeric(df['pop_migrated'])
    / pd.to_numeric(df['state_population'])

)

Add a row with the adjusted MOE

In [ ]:
df['moe_adjusted'] = (
    pd.to_numeric(df['MOE +/-'])
    / pd.to_numeric(df['state_population'])

)

Need to manually drop DC here because its input differently than in the other years -- here, the D.C. figure for folks that lived in D.C. both this year and last year is not n/a or 0, but rather a substantial number that represents the number of folks that did NOT move since last year's census

In [ ]:
df = df.drop(['District of Columbia'])

Add a column with the state names so the states arent just in an index, and reset the index to numbers

In [ ]:
df.insert(0, "state", df.index)

df = df.reset_index(drop=True)


df.head()

Looking at states with most people coming to DC

In [ ]:
df.sort_values(by='migrated_adjusted', ascending=False).head(10)


To export, want to make a slice excluding all DMV states -- Maryland, Virginia

In [ ]:
df = df[df["state"] != "Maryland"]
df = df[df["state"] != "Virginia"]
df = df[df["state"] != "Puerto Rico"]

Now add in 2008 election winners for each state

First, read in file

In [ ]:
df_2008_election_winners = pd.read_csv('pres-election-results-2008.csv')

type(df_2008_election_winners)

Next, add winning party and percent of votes columns

In [ ]:
df_2008_election_winners.drop(columns = ['Unnamed: 0'], inplace = True)

df_2008_election_winners.head()

In [ ]:

df['winning_party'] = df_2008_election_winners['party_detailed']
df['winner_percent_of_votes'] = df_2008_election_winners['percent_total_votes']

df.head()

Save as csv file

In [ ]:
df.to_csv('DC-migration-2009.csv')